In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nomio211_socialdata_path = kagglehub.dataset_download('nomio211/socialdata')

print('Data source import complete.')


Topic modeling - Extracting meaningful insights from unstructured text data

#Preprocessing


In [ ]:
!pip install fasttext
!pip install pandas

In [ ]:
import pandas as pd
import re
import fasttext

# Topic modeling


In [ ]:
!pip install sentence-transformers

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "tm_sumbee_thesis_dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "nomio211/socialdata",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
"sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

docs = df["Text"].tolist()

embeddings = model.encode(docs, show_progress_bar=True)

In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(language="multilingual")

topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.get_topic_info()

# NER

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="nomio4640/mongolian-bert-ner",
    tokenizer="nomio4640/mongolian-bert-ner",
    aggregation_strategy="simple"
)

ner("Улаанбаатар хотод Бат-Эрдэнэ ажилладаг")

In [ ]:
def clean_entities(entities):

    cleaned = []

    for ent in entities:
        word = ent["word"]

        # subword token
        if word.startswith("##") and len(cleaned) > 0:
            cleaned[-1]["word"] += word.replace("##","")
        else:
            cleaned.append(ent)

    return cleaned

In [ ]:
df["entitiesMN"] = df["clean_text"].apply(ner)

In [ ]:
df["entities_cleanMN"] = df["entitiesMN"].apply(clean_entities)

In [ ]:
from collections import Counter

# бүх entity-г нэг list болгох
all_entities = []
for row in df["entities_cleanMN"]:
    for ent in row:
        all_entities.append((ent["word"], ent["entity_group"]))

# төрөлөөр нь ялгах
per = [w for w,t in all_entities if t == "PER"]
org = [w for w,t in all_entities if t == "ORG"]
loc = [w for w,t in all_entities if t == "LOC"]

# хамгийн их давтагдсан 10
top_per = Counter(per).most_common(20)
top_org = Counter(org).most_common(20)
top_loc = Counter(loc).most_common(20)

print("Top 10 PERSON:")
for w,c in top_per:
    print(w, c)

print("\nTop 10 ORG:")
for w,c in top_org:
    print(w, c)

print("\nTop 10 LOC:")
for w,c in top_loc:
    print(w, c)

Multilingual

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple"
)

df["entities"] = df["clean_text"].apply(ner)

In [ ]:
df["entities_clean"] = df["entities"].apply(clean_entities)

In [ ]:
for row in df["entities_clean"].head(20):
    for ent in row:
        print(ent["word"], ent["entity_group"])

# Sentiment analysis

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "sumbee_thesis_dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "nomio211/socialdata",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

In [ ]:
import torch
from transformers import pipeline

# 1. Clear memory before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True, # This is the critical fix
    max_length=512,  # Standard limit for XLM-R
    device= 0         # Uses GPU (remove if you don't have one)
)

df["sentiment"] = df["Text"].apply(sentiment)

In [ ]:
# 1. Run the model on the whole column and save it to the variable 'results'
# This is what fixes the NameError
results = sentiment(df["Text"].tolist(), batch_size=16)

# 2. Extract the label and score from the 'results' list
df['sentiment_label'] = [res['label'] for res in results]
df['confidence'] = [res['score'] for res in results]

# 3. Show the result in a table
df[['Text', 'sentiment_label', 'confidence']].head(10)

In [ ]:
from google.colab import files
files.download("tweets_with_entities.xlsx")

# Network analysis


In [ ]:
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt

# бүх entity цуглуулах
all_entities = []

for row in df["entities"]:
    for ent in row:
        all_entities.append(ent["word"])

# хамгийн их давтагдсан entity
top_entities = [e for e,_ in Counter(all_entities).most_common(10)]

center = top_entities[0]

G = nx.Graph()

for row in df["entities"]:
    words = [ent["word"] for ent in row]

    if center in words:
        for w in words:
            if w != center:
                G.add_edge(center, w)

plt.figure(figsize=(8,8))

pos = nx.spring_layout(G)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=3000,
    node_color="skyblue",
    font_size=10
)

plt.title("Entity Mind Map")
plt.show()

In [ ]:
connections = Counter()

for row in df["entities"]:
    words = [ent["word"] for ent in row]

    if center in words:
        for w in words:
            if w != center:
                connections[w] += 1

top_connections = connections.most_common(10)

In [ ]:
G = nx.Graph()

for word,count in top_connections:
    G.add_edge(center, word, weight=count)

plt.figure(figsize=(8,8))

pos = nx.spring_layout(G)

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=3000,
    node_color="lightgreen"
)

plt.show()

# NER updated

In [ ]:
import re
import pandas as pd

# 1. Load the data
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("sumbee_thesis_dataset.csv")

input for kaggle

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "sumbee_thesis_dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "nomio211/socialdata",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

input for colab

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple"
)

df["entities"] = df["Text"].apply(ner)

In [ ]:
def clean_entities(entities):

    cleaned = []

    for ent in entities:
        word = ent["word"]

        # subword token
        if word.startswith("##") and len(cleaned) > 0:
            cleaned[-1]["word"] += word.replace("##","")
        else:
            cleaned.append(ent)

    return cleaned

In [ ]:
df["entities_clean"] = df["entities"].apply(clean_entities)

In [ ]:
from collections import Counter

# бүх entity-г нэг list болгох
all_entities = []
for row in df["entities_clean"]:
    for ent in row:
        all_entities.append((ent["word"], ent["entity_group"]))

# төрөлөөр нь ялгах
per = [w for w,t in all_entities if t == "PER"]
org = [w for w,t in all_entities if t == "ORG"]
loc = [w for w,t in all_entities if t == "LOC"]

# хамгийн их давтагдсан 10
top_per = Counter(per).most_common(20)
top_org = Counter(org).most_common(20)
top_loc = Counter(loc).most_common(20)

print("Top 10 PERSON:")
for w,c in top_per:
    print(w, c)

print("\nTop 10 ORG:")
for w,c in top_org:
    print(w, c)

print("\nTop 10 LOC:")
for w,c in top_loc:
    print(w, c)

# For label

In [ ]:
import re
import pandas as pd

# 1. Load the data
from google.colab import files
uploaded = files.upload()

In [ ]:
def clean_entities(entities):

    cleaned = []

    for ent in entities:
        word = ent["word"]

        # subword token
        if word.startswith("##") and len(cleaned) > 0:
            cleaned[-1]["word"] += word.replace("##","")
        else:
            cleaned.append(ent)

    return cleaned